In [ ]:
import sys
import os
from pathlib import Path
import subprocess

In [ ]:
import sys
import os
from pathlib import Path
import subprocess
import yaml

cfg_file = Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\data_for_cellvit_GS40_balanced\train_configs\ViT256\train_config.yaml")

# Load existing config
with open(cfg_file, 'r') as f:
    config = yaml.safe_load(f)

# Add all missing parameters
base_dir = r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\data_for_cellvit_GS40_balanced"

# Logging parameters
config['logging']['log_dir'] = f"{base_dir}\\logs"
config['logging']['wandb_dir'] = f"{base_dir}\\wandb"
config['logging']['checkpoint_dir'] = f"{base_dir}\\checkpoints"
config['logging']['notes'] = 'CellViT hyperparameter sweep on GS40 balanced dataset'

# Add random_seed
config['random_seed'] = 42

# Enable sweep
config['run_sweep'] = True

# Add sweep configuration
config['sweep_configuration'] = {
    'method': 'bayes',
    'metric': {
        'name': 'val_f1',
        'goal': 'maximize'
    },
    'parameters': {
        'learning_rate': {
            'distribution': 'log_uniform_values',
            'min': 1e-5,
            'max': 1e-3
        },
        'batch_size': {
            'values': [16, 32, 64]
        },
        'optimizer': {
            'values': ['adam', 'adamw', 'sgd']
        },
        'scheduler': {
            'values': ['cosine', 'step'] #constant
        },
        'unfreeze_epoch': {
            'values': [0, 5, 10]
        },
        'weight_decay': {
            'distribution': 'log_uniform_values',
            'min': 1e-6,
            'max': 1e-3
        }
    }
}

config['sweep_count'] = 5

config['training'].update({
    'epochs': 30,
    'fold': 0,
    'early_stopping_patience': 5,
})

# Save config
with open(cfg_file, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print("✅ Config saved")

# Run sweep with LIVE OUTPUT
CELLVIT_REPO = r"\\kittyserverdw\Andre_kit\data\students\Diogo\codes\CellViT_plus_plus"
CONDA_ENV = "CellViT"

train_cmd = f'conda run -n {CONDA_ENV} python {CELLVIT_REPO}\\cellvit\\train_cell_classifier_head.py --config {cfg_file}'

print(f"\n🚀 Starting sweep with live output...\n")
print("="*60)

# Run with live output (no capture_output)
result = subprocess.run(train_cmd, shell=True)

print("="*60)
if result.returncode != 0:
    print(f"\n❌ Command failed with exit code: {result.returncode}")
else:
    print("\n✅ Sweep completed!")

✅ Config saved

🚀 Starting sweep with live output...



In [ ]:
import sys
import os
from pathlib import Path
import subprocess
import yaml

cfg_file = Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\data_for_cellvit_GS40_balanced\train_configs\ViT256\train_config.yaml")

# Load existing config
with open(cfg_file, 'r') as f:
    config = yaml.safe_load(f)

# Add all missing parameters
base_dir = r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\data_for_cellvit_GS40_balanced"

# Logging parameters
config['logging']['log_dir'] = f"{base_dir}\\logs"
config['logging']['wandb_dir'] = f"{base_dir}\\wandb"
config['logging']['checkpoint_dir'] = f"{base_dir}\\checkpoints"
config['logging']['notes'] = 'CellViT hyperparameter sweep on GS40 balanced dataset'

# Add random_seed
config['random_seed'] = 42

# Enable sweep
config['run_sweep'] = True

# Add sweep configuration
config['sweep_configuration'] = {
    'method': 'bayes',  # Bayesian optimization
    'metric': {
        'name': 'val_f1',  # Optimize validation F1 score
        'goal': 'maximize'
    },
    'parameters': {
        'learning_rate': {
            'distribution': 'log_uniform_values',
            'min': 1e-5,
            'max': 1e-3
        },
        'batch_size': {
            'values': [16, 32, 64]
        },
        'optimizer': {
            'values': ['adam', 'adamw', 'sgd']
        },
        'scheduler': {
            'values': ['constant', 'cosine', 'step']
        },
        'unfreeze_epoch': {
            'values': [0, 5, 10]
        },
        'weight_decay': {
            'distribution': 'log_uniform_values',
            'min': 1e-6,
            'max': 1e-3
        }
    }
}

# Set sweep count
config['sweep_count'] = 5  # Try 20 different configurations

# Base training parameters
config['training'].update({
    'epochs': 30,
    'fold': 0,
    'early_stopping_patience': 5,
})

# Save back
with open(cfg_file, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print("✅ Config updated for hyperparameter sweep")
print(f"   run_sweep: {config['run_sweep']}")
print(f"   sweep_method: {config['sweep_configuration']['method']}")
print(f"   sweep_count: {config['sweep_count']} runs")
print(f"   optimizing: {config['sweep_configuration']['metric']['name']}")
print(f"   wandb user: andreforjaz")
print("\nHyperparameters to sweep:")
for param, settings in config['sweep_configuration']['parameters'].items():
    print(f"   - {param}: {settings}")

# %% Run Sweep

CELLVIT_REPO = r"\\kittyserverdw\Andre_kit\data\students\Diogo\codes\CellViT_plus_plus"
CONDA_ENV = "CellViT"
CFG = cfg_file

train_cmd = f'conda run -n {CONDA_ENV} python {CELLVIT_REPO}\\cellvit\\train_cell_classifier_head.py --config {CFG}'

print(f"\n{'='*60}")
print("🚀 STARTING HYPERPARAMETER SWEEP")
print(f"{'='*60}")
print(f"Command: {train_cmd}\n")
print("Monitor progress at: andreforjaz-johns-hopkins-university/cellvit_GS40_balanced")
print(f"{'='*60}\n")

result = subprocess.run(train_cmd, shell=True, capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print(f"\n❌ ERROR:\n{result.stderr}")
else:
    print("\n✅ Sweep completed successfully!")
    print("Check wandb dashboard for best parameters")

In [ ]:
CELLVIT_REPO = r"\\kittyserverdw\Andre_kit\data\students\Diogo\codes\CellViT_plus_plus"
CFG = r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\data_for_cellvit_GS40\train_configs\ViT256\train_config.yaml"
CONDA_ENV= 'CellViT'
# We use 'conda run' to guarantee the training starts in the correct environment
!conda run -n {CONDA_ENV} python "{CELLVIT_REPO}\\cellvit\\train_cell_classifier_head.py" --config "{CFG}"

In [2]:
import yaml
from pathlib import Path

cfg_file = Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\data_for_cellvit_GS40_balanced\train_configs\ViT256\train_config.yaml")

base_dir = Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\data_for_cellvit_GS40_balanced")

# Load label map from existing config
with open(cfg_file, 'r') as f:
    old_config = yaml.safe_load(f)
    label_map = old_config['data']['label_map']
    num_classes = old_config['data']['num_classes']

# Create proper config matching CellViT format
config = {
    'logging': {
        'mode': 'online',  # Use 'offline' if you don't want wandb syncing
        'project': 'cellvit_GS40_balanced',
        'notes': 'CellViT hyperparameter sweep on GS40 balanced fetal monkey dataset',
        'log_comment': 'single_split_20x_oversampled',
        'wandb_dir': str(base_dir / 'wandb'),
        'log_dir': str(base_dir / 'logs'),
        'level': 'Info',  # Can be 'Debug', 'Info', 'Warning', 'Error'
        'checkpoint_dir': str(base_dir / 'checkpoints')
    },
    
    'sweep': {
        'method': 'bayes',  # Bayesian optimization
        'name': 'gs40_fetal_sweep',
        'metric': {
            'goal': 'maximize',
            'name': 'F1-Score/Validation'  # Or 'AUROC/Validation' or 'F1/Validation'
        },
        'run_cap': 50  # Number of sweep runs
    },
    
    'random_seed': 42,
    'gpu': 0,
    
    'data': {
        'dataset': 'DetectionDataset',  # Or whatever CellViT uses
        'dataset_path': str(base_dir),
        'normalize_stains_train': False,
        'normalize_stains_val': False,
        'num_classes': num_classes,
        'train_filelist': str(base_dir / 'splits' / 'fold_0' / 'train.csv'),
        'val_filelist': str(base_dir / 'splits' / 'fold_0' / 'val.csv'),
        'test_filelist': str(base_dir / 'splits' / 'test.csv'),
        'label_map': label_map
    },
    
    'cellvit_path': r'\\kittyserverdw\Andre_kit\data\students\Diogo\codes\CellViT_plus_plus\checkpoints\CellViT-256-x40-AMP.pth',
    
    'model': {
        'parameters': {
            'hidden_dim': {
                'values': [128, 256, 512]
            }
        }
    },
    
    'training': {
        'cache_cell_dataset': True,
        'batch_size': 64,
        'epochs': 50,  # Fewer epochs for sweep
        'drop_rate': 0.1,
        'fold': 0,
        'optimizer': 'AdamW',
        'optimizer_hyperparameter': {
            'betas': [0.85, 0.9],
            'parameters': {
                'lr': {
                    'min': 0.000001, #0.00001
                    'max': 0.0001 #0.001
                },
                'weight_decay': {
                    'min': 0.00001,
                    'max': 0.0001
                }
            }
        },
        'early_stopping_patience': 5,
        'scheduler': {
            'parameters': {
                'scheduler_type': {
                    'values': ['constant', 'cosine', 'exponential']
                }
            }
        },
        'mixed_precision': True,
        'eval_every': 1
    }
}

# Save config
with open(cfg_file, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print("✅ Config created with proper CellViT schema")
print(f"\n📋 Configuration Summary:")
print(f"   Project: {config['logging']['project']}")
print(f"   Sweep method: {config['sweep']['method']}")
print(f"   Sweep runs: {config['sweep']['run_cap']}")
print(f"   Optimizing: {config['sweep']['metric']['name']}")
print(f"   Epochs per run: {config['training']['epochs']}")
print(f"   Batch size: {config['training']['batch_size']}")
print(f"   Dataset: {base_dir}")
print(f"   Num classes: {num_classes}")
print(f"\nHyperparameters to sweep:")
print(f"   - Learning rate: {config['training']['optimizer_hyperparameter']['parameters']['lr']}")
print(f"   - Weight decay: {config['training']['optimizer_hyperparameter']['parameters']['weight_decay']}")
print(f"   - Hidden dim: {config['model']['parameters']['hidden_dim']['values']}")
print(f"   - Scheduler: {config['training']['scheduler']['parameters']['scheduler_type']['values']}")

# Display full config
print(f"\n{'='*60}")
print("Full YAML config:")
print(f"{'='*60}")
print(yaml.dump(config, default_flow_style=False, sort_keys=False))

✅ Config created with proper CellViT schema

📋 Configuration Summary:
   Project: cellvit_GS40_balanced
   Sweep method: bayes
   Sweep runs: 50
   Optimizing: F1-Score/Validation
   Epochs per run: 50
   Batch size: 64
   Dataset: \\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\data_for_cellvit_GS40_balanced
   Num classes: 19

Hyperparameters to sweep:
   - Learning rate: {'min': 1e-06, 'max': 0.0001}
   - Weight decay: {'min': 1e-05, 'max': 0.0001}
   - Hidden dim: [128, 256, 512]
   - Scheduler: ['constant', 'cosine', 'exponential']

Full YAML config:
logging:
  mode: online
  project: cellvit_GS40_balanced
  notes: CellViT hyperparameter sweep on GS40 balanced fetal monkey dataset
  log_comment: single_split_20x_oversampled
  wandb_dir: \\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\data_for_cellvit_GS40_balanced\wandb
  log_dir: \\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\data_for_ce

In [ ]:
python "\\kittyserverdw\Andre_kit\data\students\Diogo\codes\CellViT_plus_plus\cellvit\train_cell_classifier_head.py" --config "\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\data_for_cellvit_GS40_balanced\train_configs\ViT256\train_config.yaml"